In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings

# Отключаем предупреждения
warnings.filterwarnings('ignore')

In [2]:
# Загружаем очищенные данные
df = pd.read_csv('cleaned_data.csv')

In [3]:
# Убираем все другие целевые переменные, чтобы избежать утечки данных
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI', 'CC50_class', 'SI_median_class', 'SI_8_class']
df = df.drop(columns=targets_to_drop, errors='ignore')

# Отделяем признаки (X) и целевую переменную (y)
X = df.drop(columns=['IC50_class'])
y = df['IC50_class']

# Разбиваем данные выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# Масштабируем данные
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
# Задаем модели и сетку гиперпараметров для классификаторов
models = {
    "Logistic Regression (Логистическая регрессия)": (LogisticRegression(max_iter=1000), {'C': [0.1, 1.0, 10.0]}),
    "Random Forest (Случайный лес)": (RandomForestClassifier(random_state=42), {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}),
    "Gradient Boosting (Градиентный бустинг)": (GradientBoostingClassifier(random_state=42), {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}),
    "SVC (Опорные векторы)": (SVC(probability=True, random_state=42), {'C': [1.0, 10.0, 50.0]}),
    "KNN (Ближайшие соседи)": (KNeighborsClassifier(), {'n_neighbors': [5, 10, 15]})
}

In [6]:
# Обучение и оценка моделей
results = {}

for name, (model, params) in models.items():
    # Используем ROC-AUC для выбора лучшей модели при кросс-валидации
    grid = GridSearchCV(model, params, cv=3, scoring='roc_auc', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)

    # Предсказание классов и вероятностей (для ROC-AUC) на тестовой выборке
    best_model = grid.best_estimator_
    predictions = best_model.predict(X_test_scaled)
    probabilities = best_model.predict_proba(X_test_scaled)[:, 1]

    # Расчет метрик
    acc = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    results[name] = {'Accuracy': acc, 'F1': f1, 'ROC-AUC': roc_auc, 'Best Params': grid.best_params_}

In [7]:
# Вывод результатов
print("Итоговые результаты сравнения моделей классификации (IC50_class):\n")
for name, metrics in results.items():
    print(f"Модель: {name}")
    print(f"Лучшие параметры: {metrics['Best Params']}")
    print(f"Метрика Accuracy (Точность): {metrics['Accuracy']:.3f}")
    print(f"Метрика F1-score: {metrics['F1']:.3f}")
    print(f"Метрика ROC-AUC: {metrics['ROC-AUC']:.3f}\n")

Итоговые результаты сравнения моделей классификации (IC50_class):

Модель: Logistic Regression (Логистическая регрессия)
Лучшие параметры: {'C': 0.1}
Метрика Accuracy (Точность): 0.704
Метрика F1-score: 0.712
Метрика ROC-AUC: 0.768

Модель: Random Forest (Случайный лес)
Лучшие параметры: {'max_depth': 10, 'n_estimators': 50}
Метрика Accuracy (Точность): 0.699
Метрика F1-score: 0.696
Метрика ROC-AUC: 0.798

Модель: Gradient Boosting (Градиентный бустинг)
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100}
Метрика Accuracy (Точность): 0.699
Метрика F1-score: 0.699
Метрика ROC-AUC: 0.785

Модель: SVC (Опорные векторы)
Лучшие параметры: {'C': 1.0}
Метрика Accuracy (Точность): 0.715
Метрика F1-score: 0.701
Метрика ROC-AUC: 0.809

Модель: KNN (Ближайшие соседи)
Лучшие параметры: {'n_neighbors': 10}
Метрика Accuracy (Точность): 0.726
Метрика F1-score: 0.712
Метрика ROC-AUC: 0.773



**Вывод для классификации IC50.**

В ходе классификации лучший результат по метрике ROC-AUC показала модель SVC (0.809), а наибольшую точность выдал KNN (Accuracy = 0.726). Случайный лес, бустинг и логистическая регрессия отработали примерно на одном уровне с Accuracy около 0.70.

**Применимость.** В отличие от регрессии, классификаторы показали хорошие результаты. Значение ROC-AUC около 0.80 означает, что модели вполне применимы на практике.

**Рекомендации по улучшению.** Для повышения качества моделей можно попробовать методы снижения размерности (например, PCA), чтобы убрать зашумляющие признаки. Также имеет смысл расширить сетку перебора гиперпараметров GridSearchCV (например, добавить другие типы ядер для SVC или изменить балансировку классов).